# How PCA Works: A Step-by-Step Visual Guide

This notebook walks through the PCA algorithm step-by-step using a **routine laboratory panel**.

**Scenario:** We have 10 correlated blood tests drawn from patients on a ward:
- Sodium, Potassium, Creatinine
- Urea, Glucose, WBC
- Hemoglobin, Platelets, CRP
- Lactate

Patients sit in different clinical states (Stable, Post-operative, Sepsis, Recovering) - can PCA help us see these patterns and detect results that don't fit, in 10-dimensional data?

**The Challenge:** We can't visualize 10 dimensions! PCA reduces the data to a few key dimensions while preserving the most important information.

> **A note on the data.** Every result in this notebook is simulated from published
> laboratory reference ranges. No real or identifiable patient data is used.

**The 8 Steps of PCA:**
0. Explore the data (correlation matrix - we can't plot 45 pairs!)
1. Visualize a 3D slice of the raw data
2. Standardize the data (z-score normalization)
3. Find PC1: the direction of maximum variance
4. Deflation: subtract PC1 projections
5. Find PC2: maximum variance in the remaining space
6. Deflation: subtract PC2 projections
7. Final result: new coordinate system reveals patterns

---


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
np.random.seed(42)

# Color map for patient states (defined early for use throughout).
# The four states match notebook 09, so the two PCA notebooks tell one story.
STATE_COLORS = {
    'Stable': 'royalblue',
    'Post-operative': 'mediumpurple',
    'Sepsis': 'crimson',
    'Recovering': 'mediumseagreen',
    'Anomaly': 'black'
}

print("Ready to explore how PCA works!")


Ready to explore how PCA works!


In [2]:
def generate_lab_panel_data(n_points_per_state=35, add_anomalies=True):
    """
    Generate a realistic correlated laboratory panel: 10 tests across 4 patient states.

    Tests (10 features), with typical adult reference ranges:
    0. Sodium (mmol/L)        (135-145)
    1. Potassium (mmol/L)     (3.5-5.0)
    2. Creatinine (µmol/L)    (50-110)
    3. Urea (mmol/L)          (2.5-7.5)
    4. Glucose (mmol/L)       (3.9-7.8 non-fasting)
    5. White cell count (10^9/L) (4-11)
    6. Hemoglobin (g/L)       (120-160)
    7. Platelets (10^9/L)     (150-400)
    8. CRP (mg/L)             (<5 normal, >100 marked inflammation)
    9. Lactate (mmol/L)       (0.5-2.0)

    Patient States:
    - Stable: everything close to the reference range
    - Post-operative: inflammatory response, hemoglobin down from blood loss
    - Sepsis: CRP and WBC high, lactate high, acute kidney injury, platelets consumed
    - Recovering: inflammation settling, platelets rebounding

    The tests are not independent: CRP drives WBC and lactate, lactate (poor
    perfusion) drives creatinine, creatinine drives urea and potassium, and
    inflammation pulls hemoglobin and platelets down. That redundancy is exactly
    what PCA exploits.

    Values are simulated from published reference ranges. No real or
    identifiable patient data is used.
    """
    np.random.seed(42)

    states = []
    all_data = []

    def generate_state_data(n, base_sodium, base_potassium, base_creatinine,
                            base_urea, base_glucose, base_wbc, base_hemoglobin,
                            base_platelets, base_crp, base_lactate, noise_scale=1.0):
        """Generate correlated laboratory results for one patient state."""
        noise = np.random.randn(n, 10) * noise_scale

        # CRP is the primary inflammatory driver
        crp = base_crp + noise[:, 0] * 25

        # Inflammation raises the white cell count
        wbc = base_wbc + (crp - base_crp) * 0.030 + noise[:, 1] * 1.2

        # Inflammation and poor perfusion raise lactate
        lactate = base_lactate + (crp - base_crp) * 0.012 + noise[:, 2] * 0.30

        # Poor perfusion injures the kidney
        creatinine = base_creatinine + (lactate - base_lactate) * 22 + noise[:, 3] * 13

        # Urea tracks creatinine closely
        urea = base_urea + (creatinine - base_creatinine) * 0.057 + noise[:, 4] * 1.1

        # Potassium climbs when the kidney fails to excrete it
        potassium = base_potassium + (creatinine - base_creatinine) * 0.0028 + noise[:, 5] * 0.25

        # Sodium drifts down with acute illness
        sodium = base_sodium - (crp - base_crp) * 0.010 + noise[:, 6] * 2

        # Stress hyperglycemia
        glucose = base_glucose + (crp - base_crp) * 0.0083 + noise[:, 7] * 0.67

        # Anemia and thrombocytopenia of acute illness
        hemoglobin = base_hemoglobin - (crp - base_crp) * 0.08 + noise[:, 8] * 7
        platelets = base_platelets - (crp - base_crp) * 0.50 + noise[:, 9] * 30

        return np.column_stack([sodium, potassium, creatinine, urea, glucose,
                                wbc, hemoglobin, platelets, crp, lactate])

    # Stable - everything close to the reference range
    stable_data = generate_state_data(
        n_points_per_state,
        base_sodium=139, base_potassium=4.1, base_creatinine=80, base_urea=5.0,
        base_glucose=5.8, base_wbc=7.0, base_hemoglobin=132, base_platelets=250,
        base_crp=6, base_lactate=1.0, noise_scale=0.8
    )
    all_data.append(stable_data)
    states.extend(['Stable'] * n_points_per_state)

    # Post-operative - surgical inflammatory response, hemoglobin down
    postop_data = generate_state_data(
        n_points_per_state,
        base_sodium=137, base_potassium=4.3, base_creatinine=93, base_urea=6.4,
        base_glucose=8.1, base_wbc=12.5, base_hemoglobin=105, base_platelets=210,
        base_crp=85, base_lactate=1.6, noise_scale=1.0
    )
    all_data.append(postop_data)
    states.extend(['Post-operative'] * n_points_per_state)

    # Sepsis - the extreme state: CRP, WBC, lactate and creatinine all up
    sepsis_data = generate_state_data(
        n_points_per_state,
        base_sodium=134, base_potassium=4.8, base_creatinine=186, base_urea=13.6,
        base_glucose=10.0, base_wbc=19.0, base_hemoglobin=98, base_platelets=130,
        base_crp=210, base_lactate=4.5, noise_scale=1.2
    )
    all_data.append(sepsis_data)
    states.extend(['Sepsis'] * n_points_per_state)

    # Recovering - inflammation settling, platelets rebounding
    recovering_data = generate_state_data(
        n_points_per_state,
        base_sodium=138, base_potassium=4.2, base_creatinine=106, base_urea=7.1,
        base_glucose=6.7, base_wbc=9.0, base_hemoglobin=110, base_platelets=320,
        base_crp=45, base_lactate=1.2, noise_scale=1.0
    )
    all_data.append(recovering_data)
    states.extend(['Recovering'] * n_points_per_state)

    data = np.vstack(all_data)
    states = np.array(states)

    # Add anomalies if requested
    anomaly_mask = np.zeros(len(data), dtype=bool)
    if add_anomalies:
        # Results that violate the normal correlation structure without being
        # absurd. Each is a picture a clinician would notice immediately.
        anomaly_data = [
            # Anomaly 1: "Isolated renal failure" - creatinine, urea and potassium
            # are high, but CRP, WBC and lactate are completely normal. The usual
            # pattern says a creatinine of 400 µmol/L comes with inflammation; this
            # doesn't (obstruction, or a nephrotoxic drug).
            # [Na, K, Creat, Urea, Gluc, WBC, Hgb, Plt, CRP, Lact]
            [137, 6.2, 400, 22, 6.1, 7.5, 128, 245, 8, 1.1],

            # Anomaly 2: "Occult bleed" - hemoglobin and platelets are very low
            # and lactate is up, yet CRP, WBC and creatinine are normal. The usual
            # pattern says low counts come with high inflammation; these don't.
            [138, 4.0, 88, 5.4, 5.6, 8.0, 68, 55, 12, 3.8],
        ]

        anomaly_data = np.array(anomaly_data)
        data = np.vstack([data, anomaly_data])
        states = np.append(states, ['Anomaly'] * len(anomaly_data))
        anomaly_mask = np.append(anomaly_mask, np.ones(len(anomaly_data), dtype=bool))

    # Keep every result inside a physiologically possible range
    limits = [(120, 160), (2.5, 7.0), (25, 710), (1.5, 45), (2.8, 25),
              (1, 40), (50, 180), (10, 600), (0, 350), (0.4, 15)]
    for i, (lo, hi) in enumerate(limits):
        data[:, i] = np.clip(data[:, i], lo, hi)

    return data, states, anomaly_mask

# Feature names for global access
FEATURE_NAMES = [
    'Sodium (mmol/L)', 'Potassium (mmol/L)', 'Creatinine (µmol/L)',
    'Urea (mmol/L)', 'Glucose (mmol/L)', 'WBC (10^9/L)',
    'Hemoglobin (g/L)', 'Platelets (10^9/L)', 'CRP (mg/L)',
    'Lactate (mmol/L)'
]

# Generate the demo data
data, states, anomaly_mask = generate_lab_panel_data()
print(f"Generated {len(data)} data points with {data.shape[1]} features")
print("  (simulated laboratory results - no real patient data)")
print(f"\nPatient states:")
for state in np.unique(states):
    count = np.sum(states == state)
    print(f"  {state}: {count} points")
print(f"\nTests and their ranges:")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {name}: {data[:,i].min():.1f} - {data[:,i].max():.1f}")


Generated 142 data points with 10 features
  (simulated laboratory results - no real patient data)

Patient states:
  Anomaly: 2 points
  Post-operative: 35 points
  Recovering: 35 points
  Sepsis: 35 points
  Stable: 35 points

Tests and their ranges:
  Sodium (mmol/L): 130.6 - 142.9
  Potassium (mmol/L): 3.6 - 6.2
  Creatinine (µmol/L): 47.3 - 400.0
  Urea (mmol/L): 2.3 - 22.0
  Glucose (mmol/L): 4.6 - 12.0
  WBC (10^9/L): 5.7 - 21.6
  Hemoglobin (g/L): 68.0 - 143.2
  Platelets (10^9/L): 55.0 - 401.2
  CRP (mg/L): 0.0 - 285.8
  Lactate (mmol/L): 0.4 - 5.7


---

## Step 0: Exploring 10-Dimensional Data

With 10 tests, we have 45 possible pairwise plots! Let's start with a **correlation matrix** to understand how the tests relate to each other, then look at a few selected 2D views.


In [3]:
def plot_correlation_matrix(data, feature_names):
    """Plot correlation matrix heatmap for all features."""
    # Calculate correlation matrix
    corr_matrix = np.corrcoef(data.T)

    # Create short labels for display
    short_names = ['Na', 'K', 'Creat', 'Urea', 'Gluc',
                   'WBC', 'Hgb', 'Plt', 'CRP', 'Lact']

    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix,
        x=short_names,
        y=short_names,
        colorscale='RdBu_r',
        zmid=0,
        zmin=-1,
        zmax=1,
        text=np.round(corr_matrix, 2),
        texttemplate='%{text}',
        textfont={"size": 10},
        hovertemplate='%{x} vs %{y}<br>Correlation: %{z:.2f}<extra></extra>'
    ))

    fig.update_layout(
        title=dict(
            text='Lab Panel Correlation Matrix<br><sub>Red = positive correlation, Blue = negative</sub>',
            font=dict(size=16)
        ),
        width=700,
        height=600,
        xaxis=dict(tickangle=45),
    )

    fig.show()

    print("Correlation Insights:")
    print("=" * 60)
    print("• Strong positive (red): CRP, WBC, lactate, creatinine, urea, potassium")
    print("• These tests are partly redundant - they track the same underlying illness")
    print("• Hemoglobin and platelets move the opposite way (blue) - they fall as inflammation rises")
    print("• PCA can exploit these correlations to reduce 10D → 2-3D!")

plot_correlation_matrix(data, FEATURE_NAMES)


Correlation Insights:
• Strong positive (red): CRP, WBC, lactate, creatinine, urea, potassium
• These tests are partly redundant - they track the same underlying illness
• Hemoglobin and platelets move the opposite way (blue) - they fall as inflammation rises
• PCA can exploit these correlations to reduce 10D → 2-3D!


In [4]:
# ============================================================
# CONFIGURE: Select 3 features to plot pairwise (indices 0-9)
# ============================================================
# Feature indices:
#   0: Sodium (mmol/L)       5: WBC (10^9/L)
#   1: Potassium (mmol/L)    6: Hemoglobin (g/L)
#   2: Creatinine (µmol/L)    7: Platelets (10^9/L)
#   3: Urea (mmol/L)         8: CRP (mg/L)
#   4: Glucose (mmol/L)       9: Lactate (mmol/L)

SELECTED_FEATURES = [2, 5, 8]  # <-- Change these to explore different tests!

def plot_step_0_pairwise(data, states, feature_indices, show_labels=True):
    """Step 0: Show 2D pairwise scatter plots for selected features."""

    # Get feature names
    names = [FEATURE_NAMES[i] for i in feature_indices]

    # Define the three pairs from selected features
    pairs = [
        (feature_indices[0], feature_indices[1], names[0], names[1]),
        (feature_indices[0], feature_indices[2], names[0], names[2]),
        (feature_indices[1], feature_indices[2], names[1], names[2])
    ]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[f'{p[2][:15]} vs {p[3][:15]}' for p in pairs],
        horizontal_spacing=0.08
    )

    # Track traces for toggle functionality
    state_trace_indices = {state: [] for state in STATE_COLORS.keys()}
    unlabeled_trace_indices = []

    for col, (i, j, xlabel, ylabel) in enumerate(pairs, 1):
        # Plot each state with different colors
        for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
            mask = states == state
            if not np.any(mask):
                continue

            state_trace_indices[state].append(len(fig.data))
            fig.add_trace(
                go.Scatter(
                    x=data[mask, i],
                    y=data[mask, j],
                    mode='markers',
                    marker=dict(size=8, color=STATE_COLORS[state], opacity=0.7),
                    name=state,
                    legendgroup=state,
                    showlegend=(col == 1),
                    visible=show_labels
                ),
                row=1, col=col
            )

        # Add unlabeled version (all points same color)
        unlabeled_trace_indices.append(len(fig.data))
        fig.add_trace(
            go.Scatter(
                x=data[:, i],
                y=data[:, j],
                mode='markers',
                marker=dict(size=8, color='steelblue', opacity=0.6),
                name='All Points',
                legendgroup='all',
                showlegend=(col == 1),
                visible=not show_labels
            ),
            row=1, col=col
        )

        # Update axis labels
        fig.update_xaxes(title_text=xlabel[:20], row=1, col=col)
        fig.update_yaxes(title_text=ylabel[:20], row=1, col=col)

    # Create visibility arrays for toggle
    n_traces = len(fig.data)

    labeled_visibility = [True] * n_traces
    for idx in unlabeled_trace_indices:
        labeled_visibility[idx] = False

    unlabeled_visibility = [False] * n_traces
    for idx in unlabeled_trace_indices:
        unlabeled_visibility[idx] = True

    fig.update_layout(
        title=dict(
            text='Step 0: Pairwise 2D Views (Selected Tests)<br><sub>Each plot shows only 2 of 10 dimensions - information is lost!</sub>',
            font=dict(size=16)
        ),
        width=1100,
        height=400,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.12,
            xanchor="center",
            x=0.5
        ),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                buttons=[
                    dict(args=[{"visible": labeled_visibility}], label="Show Labels", method="restyle"),
                    dict(args=[{"visible": unlabeled_visibility}], label="Hide Labels", method="restyle")
                ],
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.0, xanchor="left", y=1.25, yanchor="top"
            )
        ]
    )

    fig.show()

    print(f"Showing tests: {names}")
    print("Try changing SELECTED_FEATURES above to explore different combinations!")

plot_step_0_pairwise(data, states, SELECTED_FEATURES)


Showing tests: ['Creatinine (µmol/L)', 'WBC (10^9/L)', 'CRP (mg/L)']
Try changing SELECTED_FEATURES above to explore different combinations!


---

## Step 1: Raw Data - A 3D Slice

We can't visualize 10 dimensions, but we can look at a 3D slice using three selected tests: **Creatinine**, **Hemoglobin**, and **Platelets**.

Notice:
- The data is **not centered** at the origin
- The points form clusters based on patient state
- But remember: we're missing 7 other dimensions of information!


In [5]:
import pandas as pd
df = pd.DataFrame(data, columns=FEATURE_NAMES)
display(df)

,Sodium (mmol/L),Potassium (mmol/L),Creatinine (µmol/L),Urea (mmol/L),Glucose (mmol/L),WBC (10^9/L),Hemoglobin (g/L),Platelets (10^9/L),CRP (mg/L),Lactate (mmol/L)
0,141.427398,4.114442,101.881957,6.041217,6.293800,7.165295,128.576201,258.054300,15.934283,1.274657
1,137.472154,3.928554,58.932601,2.281231,5.891509,6.274849,127.656533,220.738888,0.000000,0.946851
2,136.865281,4.103362,73.277793,4.137777,6.244672,7.662644,126.291385,228.342862,35.312975,1.367962
3,139.454523,3.815936,65.751530,4.911677,4.649734,8.417163,125.524889,260.741736,0.000000,0.852351
4,138.115285,3.956471,80.157002,3.707850,6.489203,7.607593,132.742716,200.302370,20.769332,1.149476
...,...,...,...,...,...,...,...,...,...,...
137,136.705091,4.060387,74.827710,6.941467,7.360850,10.924341,106.106583,376.942117,40.676349,0.736546
138,140.434001,4.305170,116.838251,8.534085,7.886882,8.922951,108.868177,288.923390,61.948147,2.050570
139,135.190754,4.296848,124.099131,7.729797,6.144463,8.531721,117.482953,294.543520,56.499304,1.942008
140,137.000000,6.200000,400.000000,22.000000,6.100000,7.500000,128.000000,245.000000,8.000000,1.100000


In [6]:
# ============================================================
# CONFIGURE: Select 3 features for 3D visualization (indices 0-9)
# ============================================================
SELECTED_3D_FEATURES = [2, 6, 7]  # Creatinine, Hemoglobin, Platelets

# Get feature names for axis labels
AXIS_LABELS = {
    'x': FEATURE_NAMES[SELECTED_3D_FEATURES[0]],
    'y': FEATURE_NAMES[SELECTED_3D_FEATURES[1]],
    'z': FEATURE_NAMES[SELECTED_3D_FEATURES[2]]
}

def get_state_color(state_array):
    """Map states to colors."""
    return [STATE_COLORS.get(s, 'gray') for s in state_array]

def add_feature_axes(fig, data, feature_indices, opacity=0.3):
    """Add test coordinate axes to a 3D figure, originating from (0,0,0)."""
    # Calculate axis extents based on data range for selected features
    x_max = data[:, feature_indices[0]].max() * 1.1
    y_max = data[:, feature_indices[1]].max() * 1.1
    z_max = data[:, feature_indices[2]].max() * 1.1
    
    # X axis
    fig.add_trace(go.Scatter3d(
        x=[0, x_max], y=[0, 0], z=[0, 0],
        mode='lines',
        line=dict(color='gray', width=3),
        opacity=opacity,
        showlegend=False
    ))
    # Y axis
    fig.add_trace(go.Scatter3d(
        x=[0, 0], y=[0, y_max], z=[0, 0],
        mode='lines',
        line=dict(color='gray', width=3),
        opacity=opacity,
        showlegend=False
    ))
    # Z axis
    fig.add_trace(go.Scatter3d(
        x=[0, 0], y=[0, 0], z=[0, z_max],
        mode='lines',
        line=dict(color='gray', width=3),
        opacity=opacity,
        showlegend=False
    ))

def plot_step_1_raw_data(data, states, feature_indices=SELECTED_3D_FEATURES, show_labels=True):
    """Step 1: Show raw test data for 3 selected features by patient state."""
    
    # Extract the 3 selected features
    data_3d = data[:, feature_indices]
    
    # Calculate centroid of selected features
    centroid = data_3d.mean(axis=0)
    
    fig = go.Figure()
    
    # Add test axes from origin
    add_feature_axes(fig, data, feature_indices)
    
    # Track trace indices for toggle button
    state_traces = []
    unlabeled_trace_idx = None
    
    # Plot each patient state with different colors (for labeled view)
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        state_traces.append(len(fig.data))
        fig.add_trace(go.Scatter3d(
            x=data_3d[mask, 0], y=data_3d[mask, 1], z=data_3d[mask, 2],
            mode='markers',
            marker=dict(size=6, color=STATE_COLORS[state], opacity=0.8),
            name=f'{state} ({np.sum(mask)})',
            visible=show_labels
        ))
    
    # Add a single trace with all points in one color (for unlabeled view)
    unlabeled_trace_idx = len(fig.data)
    fig.add_trace(go.Scatter3d(
        x=data_3d[:, 0], y=data_3d[:, 1], z=data_3d[:, 2],
        mode='markers',
        marker=dict(size=6, color='steelblue', opacity=0.7),
        name=f'All Points ({len(data)})',
        visible=not show_labels
    ))
    
    # Centroid
    centroid_trace_idx = len(fig.data)
    fig.add_trace(go.Scatter3d(
        x=[centroid[0]], y=[centroid[1]], z=[centroid[2]],
        mode='markers',
        marker=dict(size=12, color='gold', symbol='diamond', 
                    line=dict(color='black', width=2)),
        name=f'Centroid'
    ))
    
    # Create visibility arrays for toggle button
    n_traces = len(fig.data)
    labeled_visibility = [True] * n_traces
    labeled_visibility[unlabeled_trace_idx] = False
    unlabeled_visibility = [True] * n_traces
    for idx in state_traces:
        unlabeled_visibility[idx] = False
    unlabeled_visibility[unlabeled_trace_idx] = True
    
    fig.update_layout(
        title=dict(
            text='Step 1: Raw Test Data (3 of 10 features)<br><sub>Different patient states form distinct clusters</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis_title=AXIS_LABELS['x'],
            yaxis_title=AXIS_LABELS['y'],
            zaxis_title=AXIS_LABELS['z'],
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98),
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                buttons=[
                    dict(args=[{"visible": labeled_visibility}], label="Show Labels", method="restyle"),
                    dict(args=[{"visible": unlabeled_visibility}], label="Hide Labels", method="restyle")
                ],
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.5, xanchor="left", y=1.15, yanchor="top"
            )
        ]
    )
    
    fig.show()
    print(f"Showing 3 of 10 features: {[FEATURE_NAMES[i] for i in feature_indices]}")
    print(f"The gold diamond shows the centroid (mean) of the selected features.")
    print(f"Notice how different patient states cluster together!")

plot_step_1_raw_data(data, states)

Showing 3 of 10 features: ['Creatinine (µmol/L)', 'Hemoglobin (g/L)', 'Platelets (10^9/L)']
The gold diamond shows the centroid (mean) of the selected features.
Notice how different patient states cluster together!


---

## Step 2: Standardizing the Data

**Why standardize?** PCA finds directions of maximum variance. If variables have different scales (Platelets in the hundreds vs Potassium around 4), the larger-scaled variable dominates unfairly.

**Standardization (z-score):**
1. Subtract the mean (centering)
2. Divide by standard deviation (scaling)

This puts all variables on the same scale (mean=0, std=1).


In [7]:
def add_centered_axes(fig, data, opacity=0.4):
    """Add coordinate axes centered at origin for standardized data."""
    padding = 1.2
    x_scale = np.abs(data[:, 0]).max() * padding
    y_scale = np.abs(data[:, 1]).max() * padding
    z_scale = np.abs(data[:, 2]).max() * padding
    
    scales = {'x': x_scale, 'y': y_scale, 'z': z_scale}
    
    for axis in ['x', 'y', 'z']:
        coords = {'x': [0, 0], 'y': [0, 0], 'z': [0, 0]}
        coords[axis] = [-scales[axis], scales[axis]]
        fig.add_trace(go.Scatter3d(
            x=coords['x'], y=coords['y'], z=coords['z'],
            mode='lines',
            line=dict(color='gray', width=3),
            opacity=opacity,
            showlegend=False
        ))

def plot_step_2_standardized(data, states, feature_indices=SELECTED_3D_FEATURES):
    """Step 2: Standardize the 3 selected features."""
    
    # Extract the 3 selected features
    data_3d = data[:, feature_indices]
    feature_names_3d = [FEATURE_NAMES[i] for i in feature_indices]
    
    # Standardize the data
    scaler = StandardScaler()
    standardized_data = scaler.fit_transform(data_3d)
    
    centroid = scaler.mean_
    std_devs = scaler.scale_
    
    fig = go.Figure()
    
    # Add centered axes
    add_centered_axes(fig, standardized_data)
    
    # Standardized data colored by state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter3d(
            x=standardized_data[mask, 0], y=standardized_data[mask, 1], z=standardized_data[mask, 2],
            mode='markers',
            marker=dict(size=5, color=STATE_COLORS[state], opacity=0.7),
            name=f'{state}'
        ))
    
    # Origin marker
    fig.add_trace(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers',
        marker=dict(size=10, color='gold', symbol='diamond', 
                    line=dict(color='black', width=2)),
        name='Origin (mean)'
    ))
    
    # Calculate axis ranges - USE EQUAL RANGES to preserve perpendicularity!
    padding = 1.2
    max_range = np.abs(standardized_data).max() * padding
    x_range = y_range = z_range = max_range
    
    fig.update_layout(
        title=dict(
            text='Step 2: Standardize the Data<br><sub>Z-score normalization: subtract mean, divide by std dev</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title=f'{feature_names_3d[0]} (z-score)', range=[-x_range, x_range]),
            yaxis=dict(title=f'{feature_names_3d[1]} (z-score)', range=[-y_range, y_range]),
            zaxis=dict(title=f'{feature_names_3d[2]} (z-score)', range=[-z_range, z_range]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode='cube'
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    print(f"Standardization of 3 selected features:")
    print("=" * 60)
    for i, name in enumerate(feature_names_3d):
        print(f"  {name:25s}: mean={centroid[i]:7.2f}, std={std_devs[i]:6.2f}")
    print("\nNow all variables have mean=0 and std=1.")
    print("This ensures each test contributes equally to finding principal components!")
    
    return standardized_data, scaler

standardized_data, scaler = plot_step_2_standardized(data, states)

Standardization of 3 selected features:
  Creatinine (µmol/L)      : mean= 119.88, std= 51.33
  Hemoglobin (g/L)         : mean= 111.25, std= 14.68
  Platelets (10^9/L)       : mean= 229.75, std= 78.05

Now all variables have mean=0 and std=1.
This ensures each test contributes equally to finding principal components!


---

## Step 3: Finding PC1

**The goal:** Find the direction through the origin that captures the most variance.

**How it works:**
1. Imagine a line through the origin
2. Project all points onto that line
3. Measure the spread (variance) of the projections
4. Rotate the line to maximize this spread

The direction that maximizes spread = **PC1**

In [8]:
def plot_step_3_find_pc1(standardized_data, states):
    """Step 3: Find PC1 - the direction of maximum variance."""
    
    feature_names_3d = [FEATURE_NAMES[i] for i in SELECTED_3D_FEATURES]
    
    # Fit PCA to get the first principal component
    pca = PCA(n_components=3)
    pca.fit(standardized_data)
    
    pc1 = pca.components_[0]  # First principal component direction
    
    # Project points onto PC1
    projections_pc1 = standardized_data @ pc1  # Scalar projections
    projected_points = np.outer(projections_pc1, pc1)  # Points on the PC1 line
    
    # Calculate variance explained
    var_explained = pca.explained_variance_ratio_[0] * 100
    
    fig = go.Figure()
    
    # Add centered axes
    add_centered_axes(fig, standardized_data)
    
    # Data points colored by state (faded)
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        fig.add_trace(go.Scatter3d(
            x=standardized_data[mask, 0], y=standardized_data[mask, 1], z=standardized_data[mask, 2],
            mode='markers',
            marker=dict(size=5, color=STATE_COLORS[state], opacity=0.4),
            name=f'{state}'
        ))
    
    # PC1 axis (extended line through origin)
    pc_scale = np.abs(projections_pc1).max() * 1.2
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc1[0], pc_scale*pc1[0]],
        y=[-pc_scale*pc1[1], pc_scale*pc1[1]],
        z=[-pc_scale*pc1[2], pc_scale*pc1[2]],
        mode='lines',
        line=dict(color='red', width=8),
        name=f'PC1 ({var_explained:.1f}% variance)'
    ))
    
    # Projections onto PC1 line
    fig.add_trace(go.Scatter3d(
        x=projected_points[:, 0],
        y=projected_points[:, 1],
        z=projected_points[:, 2],
        mode='markers',
        marker=dict(size=4, color='red', opacity=0.5),
        name='Projections onto PC1'
    ))
    
    # Draw some projection lines (subset for clarity)
    for i in range(0, len(standardized_data), 8):
        fig.add_trace(go.Scatter3d(
            x=[standardized_data[i, 0], projected_points[i, 0]],
            y=[standardized_data[i, 1], projected_points[i, 1]],
            z=[standardized_data[i, 2], projected_points[i, 2]],
            mode='lines',
            line=dict(color='pink', width=1),
            showlegend=False
        ))
    
    # Calculate axis ranges - USE EQUAL RANGES to preserve perpendicularity!
    padding = 1.2
    max_range = np.abs(standardized_data).max() * padding
    x_range = y_range = z_range = max_range
    
    fig.update_layout(
        title=dict(
            text=f'Step 3: Find PC1<br><sub>The direction that captures the most spread ({var_explained:.1f}% of variance)</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title=f'{feature_names_3d[0]} (z)', range=[-x_range, x_range]),
            yaxis=dict(title=f'{feature_names_3d[1]} (z)', range=[-y_range, y_range]),
            zaxis=dict(title=f'{feature_names_3d[2]} (z)', range=[-z_range, z_range]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode='cube'
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    print(f"PC1 direction: ({pc1[0]:.3f}, {pc1[1]:.3f}, {pc1[2]:.3f})")
    print(f"Variance captured by PC1: {var_explained:.1f}%")
    print("\nPC1 captures the main axis of variation across all patient states.")
    
    return pca, pc1, projections_pc1

pca_model, pc1, projections_pc1 = plot_step_3_find_pc1(standardized_data, states)

PC1 direction: (0.595, -0.564, -0.573)
Variance captured by PC1: 63.8%

PC1 captures the main axis of variation across all patient states.


---

## Step 4: Deflation

**The key insight:** After finding PC1, we "remove" what it captured.

**How it works:**
- For each point, subtract its projection onto PC1
- This leaves only the component perpendicular to PC1
- The data is now "flattened" into the plane perpendicular to PC1

This is called **deflation** - we deflate the variance along PC1.

In [9]:
def plot_step_4_deflation(standardized_data, states, pc1, projections_pc1):
    """Step 4: Deflation - subtract PC1 projections."""
    
    feature_names_3d = [FEATURE_NAMES[i] for i in SELECTED_3D_FEATURES]
    
    # Calculate projected points (on PC1 line)
    projected_points = np.outer(projections_pc1, pc1)
    
    # Deflation: subtract projections
    deflated_data = standardized_data - projected_points
    
    fig = go.Figure()
    
    # Add centered axes
    add_centered_axes(fig, standardized_data)
    
    # --- ADD PERPENDICULAR PLANE ---
    # Create a plane perpendicular to PC1 passing through the origin
    # Find two vectors orthogonal to PC1 to span the plane
    if abs(pc1[0]) < 0.9:
        v1 = np.array([1, 0, 0])
    else:
        v1 = np.array([0, 1, 0])
    
    # First orthogonal vector (Gram-Schmidt)
    ortho1 = v1 - np.dot(v1, pc1) * pc1
    ortho1 = ortho1 / np.linalg.norm(ortho1)
    
    # Second orthogonal vector
    ortho2 = np.cross(pc1, ortho1)
    ortho2 = ortho2 / np.linalg.norm(ortho2)
    
    # Create grid for the plane
    plane_size = np.abs(deflated_data).max() * 1.5
    u = np.linspace(-plane_size, plane_size, 20)
    v = np.linspace(-plane_size, plane_size, 20)
    U, V = np.meshgrid(u, v)
    
    # Points on the plane: P = u*ortho1 + v*ortho2
    plane_x = U * ortho1[0] + V * ortho2[0]
    plane_y = U * ortho1[1] + V * ortho2[1]
    plane_z = U * ortho1[2] + V * ortho2[2]
    
    # Add the perpendicular plane
    # fig.add_trace(go.Surface(
    #     x=plane_x, y=plane_y, z=plane_z,
    #     opacity=0.2,
    #     colorscale=[[0, 'lightblue'], [1, 'lightblue']],
    #     showscale=False,
    #     name='Perpendicular plane'
    # ))
    
    # Show standardized data (faded)
    fig.add_trace(go.Scatter3d(
        x=standardized_data[:, 0], y=standardized_data[:, 1], z=standardized_data[:, 2],
        mode='markers',
        marker=dict(size=3, color='lightgray', opacity=0.2),
        name='Before deflation'
    ))
    
    # Deflated data colored by state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter3d(
            x=deflated_data[mask, 0], y=deflated_data[mask, 1], z=deflated_data[mask, 2],
            mode='markers',
            marker=dict(size=5, color=STATE_COLORS[state], opacity=0.7),
            name=f'{state}'
        ))
    
    # PC1 axis (dashed)
    pc_scale = np.abs(projections_pc1).max() * 1.2
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc1[0], pc_scale*pc1[0]],
        y=[-pc_scale*pc1[1], pc_scale*pc1[1]],
        z=[-pc_scale*pc1[2], pc_scale*pc1[2]],
        mode='lines',
        line=dict(color='red', width=6, dash='dash'),
        name='PC1 (variance removed)'
    ))
    
    # Calculate axis ranges - USE EQUAL RANGES to preserve perpendicularity!
    padding = 1.2
    max_range = np.abs(standardized_data).max() * padding
    x_range = y_range = z_range = max_range
    
    # Camera: view from along PC1 to see the disk edge-on
    cam_pos = pc1 * 2.5 + np.array([0.3, 0.3, 0.2])
    
    fig.update_layout(
        title=dict(
            text='Step 4: Deflation<br><sub>Data flattens onto blue plane (perpendicular to PC1)</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title=f'{feature_names_3d[0]} (z)', range=[-x_range, x_range]),
            yaxis=dict(title=f'{feature_names_3d[1]} (z)', range=[-y_range, y_range]),
            zaxis=dict(title=f'{feature_names_3d[2]} (z)', range=[-z_range, z_range]),
            camera=dict(eye=dict(x=cam_pos[0], y=cam_pos[1], z=cam_pos[2])),
            aspectmode='cube'
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    # Verify perpendicularity
    dot_products = deflated_data @ pc1
    print(f"Verification: deflated · PC1 = {dot_products.mean():.2e} (should be ~0)")
    
    remaining_var = np.var(deflated_data)
    original_var = np.var(standardized_data)
    print(f"Original variance: {original_var:.2f}")
    print(f"Remaining variance: {remaining_var:.2f} ({(1 - remaining_var/original_var)*100:.1f}% removed)")
    print("\nDeflated data lies ON the blue plane, perpendicular to PC1 (red dashed).")
    
    return deflated_data

deflated_data = plot_step_4_deflation(standardized_data, states, pc1, projections_pc1)

Verification: deflated · PC1 = -7.56e-18 (should be ~0)
Original variance: 1.00
Remaining variance: 0.36 (63.8% removed)

Deflated data lies ON the blue plane, perpendicular to PC1 (red dashed).


---

## Step 5: Finding PC2

Now we repeat the process on the deflated data:
- Find the direction of maximum variance in the remaining data
- This direction is **automatically perpendicular to PC1**

Why perpendicular? Because we removed all variance along PC1!

In [10]:
def plot_step_5_find_pc2(standardized_data, deflated_data, states, pca_model):
    """Step 5: Find PC2 in the deflated data."""
    
    feature_names_3d = [FEATURE_NAMES[i] for i in SELECTED_3D_FEATURES]
    
    pc1 = pca_model.components_[0]
    pc2 = pca_model.components_[1]
    
    # Project deflated data onto PC2
    projections_pc2 = deflated_data @ pc2
    projected_points_pc2 = np.outer(projections_pc2, pc2)
    
    var_pc1 = pca_model.explained_variance_ratio_[0] * 100
    var_pc2 = pca_model.explained_variance_ratio_[1] * 100
    
    fig = go.Figure()
    
    # Add centered axes
    add_centered_axes(fig, standardized_data)
    
    # Deflated data colored by state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter3d(
            x=deflated_data[mask, 0], y=deflated_data[mask, 1], z=deflated_data[mask, 2],
            mode='markers',
            marker=dict(size=5, color=STATE_COLORS[state], opacity=0.5),
            name=f'{state}'
        ))
    
    # PC1 axis (dashed - already used)
    pc_scale = 4
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc1[0], pc_scale*pc1[0]],
        y=[-pc_scale*pc1[1], pc_scale*pc1[1]],
        z=[-pc_scale*pc1[2], pc_scale*pc1[2]],
        mode='lines',
        line=dict(color='red', width=4, dash='dash'),
        name=f'PC1 ({var_pc1:.1f}%)'
    ))
    
    # PC2 axis
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc2[0], pc_scale*pc2[0]],
        y=[-pc_scale*pc2[1], pc_scale*pc2[1]],
        z=[-pc_scale*pc2[2], pc_scale*pc2[2]],
        mode='lines',
        line=dict(color='green', width=8),
        name=f'PC2 ({var_pc2:.1f}%)'
    ))
    
    # Projections onto PC2
    fig.add_trace(go.Scatter3d(
        x=projected_points_pc2[:, 0],
        y=projected_points_pc2[:, 1],
        z=projected_points_pc2[:, 2],
        mode='markers',
        marker=dict(size=4, color='green', opacity=0.5),
        name='Projections onto PC2'
    ))
    
    # Calculate axis ranges - USE EQUAL RANGES to preserve perpendicularity!
    padding = 1.2
    max_range = np.abs(standardized_data).max() * padding
    x_range = y_range = z_range = max_range
    
    fig.update_layout(
        title=dict(
            text=f'Step 5: Find PC2<br><sub>Maximum variance in the remaining disk ({var_pc2:.1f}%)</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title=f'{feature_names_3d[0]} (z)', range=[-x_range, x_range]),
            yaxis=dict(title=f'{feature_names_3d[1]} (z)', range=[-y_range, y_range]),
            zaxis=dict(title=f'{feature_names_3d[2]} (z)', range=[-z_range, z_range]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode='cube'
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    # Verify orthogonality
    dot_product = np.dot(pc1, pc2)
    print(f"PC2 direction: ({pc2[0]:.3f}, {pc2[1]:.3f}, {pc2[2]:.3f})")
    print(f"\nPC1 · PC2 = {dot_product:.6f} (≈0 confirms they're perpendicular)")
    print(f"\nVariance captured:")
    print(f"  PC1: {var_pc1:.1f}%")
    print(f"  PC2: {var_pc2:.1f}%")
    print(f"  Total so far: {var_pc1 + var_pc2:.1f}%")

plot_step_5_find_pc2(standardized_data, deflated_data, states, pca_model)

PC2 direction: (0.102, 0.760, -0.642)

PC1 · PC2 = -0.000000 (≈0 confirms they're perpendicular)

Variance captured:
  PC1: 63.8%
  PC2: 19.6%
  Total so far: 83.4%


---

## Step 6: Deflation Again

Now we deflate again - subtracting the PC2 projections:
- The disk becomes a **line** along PC3
- Only the smallest variance remains
- This completes the pattern: football → disk → line

In [11]:
def plot_step_6_deflation_pc2(standardized_data, deflated_data, states, pca_model):
    """Step 6: Deflation again - subtract PC2 projections."""
    
    feature_names_3d = [FEATURE_NAMES[i] for i in SELECTED_3D_FEATURES]
    
    pc1 = pca_model.components_[0]
    pc2 = pca_model.components_[1]
    pc3 = pca_model.components_[2]
    
    var_pc1 = pca_model.explained_variance_ratio_[0] * 100
    var_pc2 = pca_model.explained_variance_ratio_[1] * 100
    var_pc3 = pca_model.explained_variance_ratio_[2] * 100
    
    # Project onto PC2 and subtract
    projections_pc2 = deflated_data @ pc2
    projected_points_pc2 = np.outer(projections_pc2, pc2)
    
    # Second deflation
    deflated_data_2 = deflated_data - projected_points_pc2
    
    fig = go.Figure()
    
    # Add centered axes
    add_centered_axes(fig, standardized_data)
    
    # Show disk data (faded)
    fig.add_trace(go.Scatter3d(
        x=deflated_data[:, 0], y=deflated_data[:, 1], z=deflated_data[:, 2],
        mode='markers',
        marker=dict(size=3, color='lightgray', opacity=0.2),
        name='Before (disk)'
    ))
    
    # Deflated data (now a line) colored by state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter3d(
            x=deflated_data_2[mask, 0], y=deflated_data_2[mask, 1], z=deflated_data_2[mask, 2],
            mode='markers',
            marker=dict(size=5, color=STATE_COLORS[state], opacity=0.7),
            name=f'{state}'
        ))
    
    # PC axes
    pc_scale = 3
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc1[0], pc_scale*pc1[0]],
        y=[-pc_scale*pc1[1], pc_scale*pc1[1]],
        z=[-pc_scale*pc1[2], pc_scale*pc1[2]],
        mode='lines',
        line=dict(color='red', width=4, dash='dash'),
        name=f'PC1 ({var_pc1:.1f}%)'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc2[0], pc_scale*pc2[0]],
        y=[-pc_scale*pc2[1], pc_scale*pc2[1]],
        z=[-pc_scale*pc2[2], pc_scale*pc2[2]],
        mode='lines',
        line=dict(color='green', width=4, dash='dash'),
        name=f'PC2 ({var_pc2:.1f}%) - removed'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc3[0], pc_scale*pc3[0]],
        y=[-pc_scale*pc3[1], pc_scale*pc3[1]],
        z=[-pc_scale*pc3[2], pc_scale*pc3[2]],
        mode='lines',
        line=dict(color='blue', width=8),
        name=f'PC3 ({var_pc3:.1f}%) - remaining'
    ))
    
    # Draw some arrows showing the subtraction
    # for i in range(0, len(deflated_data), 12):
    #     fig.add_trace(go.Scatter3d(
    #         x=[deflated_data[i, 0], deflated_data_2[i, 0]],
    #         y=[deflated_data[i, 1], deflated_data_2[i, 1]],
    #         z=[deflated_data[i, 2], deflated_data_2[i, 2]],
    #         mode='lines',
    #         line=dict(color='purple', width=2),
    #         showlegend=False
    #     ))
    
    # Calculate axis ranges - USE EQUAL RANGES to preserve perpendicularity!
    padding = 1.2
    max_range = np.abs(standardized_data).max() * padding
    x_range = y_range = z_range = max_range
    
    fig.update_layout(
        title=dict(
            text='Step 6: Deflation Again<br><sub>Subtract PC2 projections - the disk collapses into a line</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title=f'{feature_names_3d[0]} (z)', range=[-x_range, x_range]),
            yaxis=dict(title=f'{feature_names_3d[1]} (z)', range=[-y_range, y_range]),
            zaxis=dict(title=f'{feature_names_3d[2]} (z)', range=[-z_range, z_range]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode='cube'
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    print(f"Shape progression: 3D cloud → 2D disk → 1D line")
    print(f"\nVariance remaining: {var_pc3:.1f}% (only PC3)")
    print(f"All patient states are now collapsed along PC3 - the noise dimension.")
    
    return deflated_data_2

deflated_data_2 = plot_step_6_deflation_pc2(standardized_data, deflated_data, states, pca_model)

Shape progression: 3D cloud → 2D disk → 1D line

Variance remaining: 16.6% (only PC3)
All patient states are now collapsed along PC3 - the noise dimension.


---

## Step 7: The Final Result

After finding all principal components, we have a new coordinate system:
- PC1, PC2, PC3 form orthogonal axes
- They're ordered by how much variance they capture
- The new axes are aligned with the "natural shape" of the data

This is the essence of PCA: **finding a coordinate system that aligns with your data's structure.**

In [12]:
def plot_step_7_final_result(standardized_data, states, pca_model):
    """Step 7: Show the final PCA result with all three axes."""
    
    feature_names_3d = [FEATURE_NAMES[i] for i in SELECTED_3D_FEATURES]
    
    pc1 = pca_model.components_[0]
    pc2 = pca_model.components_[1]
    pc3 = pca_model.components_[2]
    
    var1 = pca_model.explained_variance_ratio_[0] * 100
    var2 = pca_model.explained_variance_ratio_[1] * 100
    var3 = pca_model.explained_variance_ratio_[2] * 100
    
    fig = go.Figure()
    
    # Add centered axes (faded for comparison)
    add_centered_axes(fig, standardized_data, opacity=0.2)
    
    # Data colored by state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter3d(
            x=standardized_data[mask, 0], y=standardized_data[mask, 1], z=standardized_data[mask, 2],
            mode='markers',
            marker=dict(size=6, color=STATE_COLORS[state], opacity=0.7),
            name=f'{state}'
        ))
    
    # PC axes
    pc_scale = 4
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc1[0], pc_scale*pc1[0]],
        y=[-pc_scale*pc1[1], pc_scale*pc1[1]],
        z=[-pc_scale*pc1[2], pc_scale*pc1[2]],
        mode='lines',
        line=dict(color='red', width=8),
        name=f'PC1 ({var1:.1f}%)'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc2[0], pc_scale*pc2[0]],
        y=[-pc_scale*pc2[1], pc_scale*pc2[1]],
        z=[-pc_scale*pc2[2], pc_scale*pc2[2]],
        mode='lines',
        line=dict(color='green', width=6),
        name=f'PC2 ({var2:.1f}%)'
    ))
    
    fig.add_trace(go.Scatter3d(
        x=[-pc_scale*pc3[0], pc_scale*pc3[0]],
        y=[-pc_scale*pc3[1], pc_scale*pc3[1]],
        z=[-pc_scale*pc3[2], pc_scale*pc3[2]],
        mode='lines',
        line=dict(color='blue', width=4),
        name=f'PC3 ({var3:.1f}%)'
    ))
    
    # Calculate axis ranges
    padding = 1.2
    x_range = np.abs(standardized_data[:, 0]).max() * padding
    y_range = np.abs(standardized_data[:, 1]).max() * padding
    z_range = np.abs(standardized_data[:, 2]).max() * padding
    
    fig.update_layout(
        title=dict(
            text='Step 7: The New Coordinate System<br><sub>PC1, PC2, PC3 axes aligned with patient state patterns</sub>',
            font=dict(size=18)
        ),
        scene=dict(
            xaxis=dict(title=f'{feature_names_3d[0]} (z)', range=[-x_range, x_range]),
            yaxis=dict(title=f'{feature_names_3d[1]} (z)', range=[-y_range, y_range]),
            zaxis=dict(title=f'{feature_names_3d[2]} (z)', range=[-z_range, z_range]),
            camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)),
            aspectmode='cube'
        ),
        width=900,
        height=700,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    print("\nFinal Summary:")
    print("=" * 50)
    print(f"PC1: {var1:.1f}% of variance (red)")
    print(f"PC2: {var2:.1f}% of variance (green)")
    print(f"PC3: {var3:.1f}% of variance (blue)")
    print(f"\nTotal: {var1 + var2 + var3:.1f}%")
    print("\nPCA found a new coordinate system aligned with the data's structure!")

plot_step_7_final_result(standardized_data, states, pca_model)



Final Summary:
PC1: 63.8% of variance (red)
PC2: 19.6% of variance (green)
PC3: 16.6% of variance (blue)

Total: 100.0%

PCA found a new coordinate system aligned with the data's structure!


---

## Interactive: Cycle Through All Steps

For live presentations: run the cell below repeatedly to step through the PCA process.

In [13]:
# State for cycling through steps
if 'pca_demo_step' not in dir():
    pca_demo_step = 0

def pca_step_by_step_demo():
    """Run this cell repeatedly to cycle through PCA steps."""
    global pca_demo_step, data, states, anomaly_mask, standardized_data, scaler, pca_model, pc1, projections_pc1, deflated_data
    
    # Ensure data exists
    if 'data' not in dir() or data is None:
        data, states, anomaly_mask = generate_lab_panel_data()
    
    step_names = [
        "Step 1: Raw Test Data (3 features)",
        "Step 2: Standardizing",
        "Step 3: Finding PC1",
        "Step 4: Deflation (3D → 2D disk)",
        "Step 5: Finding PC2",
        "Step 6: Deflation (2D → 1D line)",
        "Step 7: Final Result"
    ]
    
    print(f"\n{'='*50}")
    print(f"  {step_names[pca_demo_step]}")
    print(f"{'='*50}\n")
    
    if pca_demo_step == 0:
        plot_step_1_raw_data(data, states)
    elif pca_demo_step == 1:
        standardized_data, scaler = plot_step_2_standardized(data, states)
    elif pca_demo_step == 2:
        if 'standardized_data' not in dir():
            data_3d = data[:, SELECTED_3D_FEATURES]
            scaler = StandardScaler()
            standardized_data = scaler.fit_transform(data_3d)
        pca_model, pc1, projections_pc1 = plot_step_3_find_pc1(standardized_data, states)
    elif pca_demo_step == 3:
        if 'standardized_data' not in dir():
            data_3d = data[:, SELECTED_3D_FEATURES]
            scaler = StandardScaler()
            standardized_data = scaler.fit_transform(data_3d)
        if 'pc1' not in dir():
            pca_model = PCA(n_components=3).fit(standardized_data)
            pc1 = pca_model.components_[0]
            projections_pc1 = standardized_data @ pc1
        deflated_data = plot_step_4_deflation(standardized_data, states, pc1, projections_pc1)
    elif pca_demo_step == 4:
        if 'deflated_data' not in dir():
            pc1 = pca_model.components_[0]
            projections_pc1 = standardized_data @ pc1
            deflated_data = standardized_data - np.outer(projections_pc1, pc1)
        plot_step_5_find_pc2(standardized_data, deflated_data, states, pca_model)
    elif pca_demo_step == 5:
        if 'deflated_data' not in dir():
            pc1 = pca_model.components_[0]
            projections_pc1 = standardized_data @ pc1
            deflated_data = standardized_data - np.outer(projections_pc1, pc1)
        plot_step_6_deflation_pc2(standardized_data, deflated_data, states, pca_model)
    elif pca_demo_step == 6:
        plot_step_7_final_result(standardized_data, states, pca_model)
    
    # Advance to next step
    pca_demo_step = (pca_demo_step + 1) % 7
    print(f"\n>>> Run this cell again for: {step_names[pca_demo_step]}")

pca_step_by_step_demo()



  Step 1: Raw Test Data (3 features)



Showing 3 of 10 features: ['Creatinine (µmol/L)', 'Hemoglobin (g/L)', 'Platelets (10^9/L)']
The gold diamond shows the centroid (mean) of the selected features.
Notice how different patient states cluster together!

>>> Run this cell again for: Step 2: Standardizing


In [14]:
# Reset to step 1
pca_demo_step = 0
print("Demo reset. Run the cell above to start from Step 1.")

Demo reset. Run the cell above to start from Step 1.


---

## Summary of the 8 Steps (3-Test Demo)

| Step | Action | What You See |
|------|--------|--------------|
| 0 | **Correlation matrix + 2D plots** | Tests are correlated, clusters overlap in 2D |
| 1 | View raw data in 3D | 3D cloud with distinct clusters |
| 2 | **Standardize** (z-score) | 3D cloud (centered & scaled) |
| 3 | Find PC1 | Direction of max variance |
| 4 | Deflate (remove PC1) | 3D → 2D disk |
| 5 | Find PC2 | Direction of max variance in disk |
| 6 | Deflate (remove PC2) | 2D → 1D line |
| 7 | View new axes | All 3 PCs visible |

**Key insight:** Each deflation step removes one dimension of variance, leaving a "flatter" dataset.

**Important:** We **standardize** (not just center) because variables have different scales. Without standardization, PCA would be dominated by whichever variable has the largest numerical values - platelets in the hundreds would swamp creatinine around 1!

---

**Next:** The Bonus section below shows PCA on all **10 tests** - demonstrating real dimensionality reduction!


---

## Bonus: PCA on All 10 Tests

Now let's see the **real power of PCA** - dimensionality reduction! The demo above used only 3 tests for visualization. Here we apply PCA to all **10 laboratory tests**.

By projecting our 10-dimensional lab data onto the principal components, we can:
1. **Visualize clusters** in 2D (PC1 vs PC2) - impossible with raw 10D data!
2. **See separation** in 1D (just PC1)
3. **Detect results** that don't fit any normal pattern


In [15]:
def plot_2d_projection_3features(data, states, feature_indices=SELECTED_3D_FEATURES):
    """Project 3D data onto PC1 and PC2, colored by patient state."""
    
    # Extract only the 3 selected features
    data_3d = data[:, feature_indices]
    feature_names = [FEATURE_NAMES[i] for i in feature_indices]
    
    # Standardize the 3 features
    scaler = StandardScaler()
    data_standardized = scaler.fit_transform(data_3d)
    
    # Fit PCA on only 3 features
    pca = PCA()
    pca_scores = pca.fit_transform(data_standardized)
    
    var1 = pca.explained_variance_ratio_[0] * 100
    var2 = pca.explained_variance_ratio_[1] * 100
    
    fig = go.Figure()
    
    # Plot each state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter(
            x=pca_scores[mask, 0],
            y=pca_scores[mask, 1],
            mode='markers',
            marker=dict(size=10, color=STATE_COLORS[state], opacity=0.8),
            name=f'{state} ({np.sum(mask)})'
        ))
    
    # Add axis lines
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
    
    fig.update_layout(
        title=dict(
            text=f'2D PCA Projection: 3 Tests → 2 Dimensions<br><sub>PC1 ({var1:.1f}%) vs PC2 ({var2:.1f}%) - captures {var1+var2:.1f}% of 3D variance</sub>',
            font=dict(size=18)
        ),
        xaxis_title=f'PC1 ({var1:.1f}% variance)',
        yaxis_title=f'PC2 ({var2:.1f}% variance)',
        width=900,
        height=600,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    print(f"2D Projection of 3 Features: {feature_names}")
    print("=" * 60)
    print(f"• Using same 3 features from the step-by-step demo above")
    print(f"• PC1 + PC2 capture {var1+var2:.1f}% of the 3D variance")
    print(f"• Clusters visible, but we're only using 3 of 10 tests!")
    print(f"\n→ Next: See how much better separation we get with ALL 10 features...")

# Show 2D projection using the 3 features from the demo
plot_2d_projection_3features(data, states)

2D Projection of 3 Features: ['Creatinine (µmol/L)', 'Hemoglobin (g/L)', 'Platelets (10^9/L)']
• Using same 3 features from the step-by-step demo above
• PC1 + PC2 capture 83.4% of the 3D variance
• Clusters visible, but we're only using 3 of 10 tests!

→ Next: See how much better separation we get with ALL 10 features...


In [16]:
import plotly.express as px

def plot_2d_projection(data, states, title="2D PCA Projection"):
    """Project 10D data onto PC1 and PC2, colored by patient state."""
    
    # Standardize all 10 features
    scaler = StandardScaler()
    data_standardized = scaler.fit_transform(data)
    
    # Fit PCA on all 10 features
    pca = PCA()
    pca_scores = pca.fit_transform(data_standardized)
    
    var1 = pca.explained_variance_ratio_[0] * 100
    var2 = pca.explained_variance_ratio_[1] * 100
    
    fig = go.Figure()
    
    # Plot each state
    for state in ['Stable', 'Post-operative', 'Sepsis', 'Recovering', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        fig.add_trace(go.Scatter(
            x=pca_scores[mask, 0],
            y=pca_scores[mask, 1],
            mode='markers',
            marker=dict(size=10, color=STATE_COLORS[state], opacity=0.8),
            name=f'{state} ({np.sum(mask)})'
        ))
    
    # Add axis lines
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
    
    fig.update_layout(
        title=dict(
            text=f'2D PCA Projection: 10 Tests → 2 Dimensions<br><sub>PC1 ({var1:.1f}%) vs PC2 ({var2:.1f}%) - captures {var1+var2:.1f}% of 10D variance</sub>',
            font=dict(size=18)
        ),
        xaxis_title=f'PC1 ({var1:.1f}% variance)',
        yaxis_title=f'PC2 ({var2:.1f}% variance)',
        width=900,
        height=600,
        legend=dict(x=0.02, y=0.98)
    )
    
    fig.show()
    
    print("2D Projection Insights (10D → 2D):")
    print("=" * 60)
    print(f"• We compressed 10 tests down to just 2 numbers!")
    print(f"• PC1 separates the patient states by how acutely unwell they are")
    print(f"• PC2 captures the secondary axis: marrow suppression vs renal injury")
    print(f"• Anomalies (black dots) are clearly separated from normal clusters!")
    print(f"\nWith just 2 dimensions, we visualize {var1+var2:.1f}% of 10D information.")

# Generate fresh data and plot
data, states, anomaly_mask = generate_lab_panel_data()
plot_2d_projection(data, states)

2D Projection Insights (10D → 2D):
• We compressed 10 tests down to just 2 numbers!
• PC1 separates the patient states by how acutely unwell they are
• PC2 captures the secondary axis: marrow suppression vs renal injury
• Anomalies (black dots) are clearly separated from normal clusters!

With just 2 dimensions, we visualize 81.4% of 10D information.


In [17]:
def plot_1d_projection(data, states):
    """Project all 10D data onto PC1 only - a 1D view colored by state."""
    
    # Standardize all 10 features
    scaler = StandardScaler()
    data_standardized = scaler.fit_transform(data)
    
    # Fit PCA on all 10 features
    pca = PCA()
    pca_scores = pca.fit_transform(data_standardized)
    
    var1 = pca.explained_variance_ratio_[0] * 100
    
    fig = go.Figure()
    
    # Calculate y-jitter for each state to avoid overlap
    y_positions = {'Stable': 1, 'Recovering': 2, 'Post-operative': 3, 'Sepsis': 4, 'Anomaly': 2.5}
    
    # Plot each state as a strip
    for state in ['Stable', 'Recovering', 'Post-operative', 'Sepsis', 'Anomaly']:
        mask = states == state
        if not np.any(mask):
            continue
        
        y_base = y_positions[state]
        y_jitter = np.random.uniform(-0.15, 0.15, np.sum(mask))
        
        fig.add_trace(go.Scatter(
            x=pca_scores[mask, 0],
            y=y_base + y_jitter,
            mode='markers',
            marker=dict(size=10, color=STATE_COLORS[state], opacity=0.7),
            name=f'{state}'
        ))
    
    # Add vertical line at origin
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
    
    fig.update_layout(
        title=dict(
            text=f'1D PCA Projection: 10 Tests → 1 Number<br><sub>PC1 captures {var1:.1f}% of 10D variation!</sub>',
            font=dict(size=18)
        ),
        xaxis_title=f'PC1 Score ({var1:.1f}% variance)',
        yaxis=dict(
            tickvals=[1, 2, 3, 4],
            ticktext=['Stable', 'Recovering', 'Post-operative', 'Sepsis'],
            title='Patient State'
        ),
        width=900,
        height=400,
        legend=dict(x=0.85, y=0.98)
    )
    
    fig.show()
    
    # Calculate state means on PC1
    print("1D Projection: 10 Features → 1 PC1 Score")
    print("=" * 60)
    for state in ['Stable', 'Recovering', 'Post-operative', 'Sepsis']:
        mask = states == state
        mean_score = pca_scores[mask, 0].mean()
        print(f"  {state:15s}: PC1 = {mean_score:+.2f}")
    
    print("\nInterpretation:")
    print("  • Low PC1 = a patient whose panel looks well (Stable)")
    print("  • High PC1 = a patient whose panel looks acutely unwell (Sepsis)")
    print("  • A SINGLE number summarizes how sick 10 lab tests say the patient is!")

plot_1d_projection(data, states)


1D Projection: 10 Features → 1 PC1 Score
  Stable         : PC1 = -2.72
  Recovering     : PC1 = -1.50
  Post-operative : PC1 = -0.11
  Sepsis         : PC1 = +4.25

Interpretation:
  • Low PC1 = a patient whose panel looks well (Stable)
  • High PC1 = a patient whose panel looks acutely unwell (Sepsis)
  • A SINGLE number summarizes how sick 10 lab tests say the patient is!


---
## Anomaly Detection with PCA

One of the most powerful applications of PCA is **anomaly detection**. The key insight:

**Results that follow the usual pattern** can be reconstructed well from a few principal components.
**Results that don't** have high **reconstruction error**.

This works because PCA learns how the tests normally move together. When a panel breaks those learned relationships - a creatinine of 400 µmol/L with a completely normal CRP and white cell count, say - the reconstruction fails and the panel is flagged, even though no single result is impossible on its own.


In [18]:
def plot_anomaly_detection(data, states, anomaly_mask, n_components=2):
    """
    Demonstrate anomaly detection using PCA reconstruction error on all 10 features.
    Uses only 2 components to reconstruct 10D data - anomalies will have high error.
    """
    
    # Standardize all 10 features
    scaler = StandardScaler()
    data_standardized = scaler.fit_transform(data)
    
    # Fit PCA with limited components (for reconstruction)
    pca = PCA(n_components=n_components)
    pca_scores = pca.fit_transform(data_standardized)
    
    # Reconstruct data from reduced dimensions
    reconstructed = pca.inverse_transform(pca_scores)
    
    # Calculate reconstruction error for each point
    reconstruction_error = np.sqrt(np.sum((data_standardized - reconstructed) ** 2, axis=1))
    
    # Determine threshold (95th percentile of normal data)
    normal_mask = ~anomaly_mask
    threshold = np.percentile(reconstruction_error[normal_mask], 95)
    
    # Create bar chart of reconstruction errors
    fig = go.Figure()
    
    # Sort by reconstruction error for visualization
    sorted_indices = np.argsort(reconstruction_error)
    
    # Color by whether it's an anomaly
    colors = ['red' if anomaly_mask[i] else 'steelblue' for i in sorted_indices]
    labels = [states[i] for i in sorted_indices]
    
    fig.add_trace(go.Bar(
        x=list(range(len(reconstruction_error))),
        y=reconstruction_error[sorted_indices],
        marker_color=colors,
        text=labels,
        hovertemplate='%{text}<br>Error: %{y:.2f}<extra></extra>'
    ))
    
    # Add threshold line
    fig.add_hline(
        y=threshold, 
        line_dash="dash", 
        line_color="orange", 
        annotation_text=f"Threshold ({threshold:.2f})",
        annotation_position="right"
    )
    
    fig.update_layout(
        title=dict(
            text=f'Anomaly Detection: 10D → {n_components}D Reconstruction Error<br><sub>Anomalies (red) violate normal test correlations</sub>',
            font=dict(size=18)
        ),
        xaxis_title='Data Points (sorted by error)',
        yaxis_title='Reconstruction Error (standardized units)',
        width=900,
        height=500,
        showlegend=False
    )
    
    fig.show()
    
    # Summary statistics
    print(f"Anomaly Detection on 10 Features:")
    print("=" * 60)
    print(f"Using {n_components} principal components to reconstruct 10D data")
    print(f"Threshold: {threshold:.2f} (95th percentile of normal data)")
    print()
    
    # Detection results
    detected_anomalies = reconstruction_error > threshold
    true_positives = np.sum(detected_anomalies & anomaly_mask)
    false_positives = np.sum(detected_anomalies & ~anomaly_mask)
    
    print(f"Anomaly Detection Performance:")
    print(f"  True Positives  (detected anomalies): {true_positives}/{np.sum(anomaly_mask)}")
    print(f"  False Positives (normal flagged):     {false_positives}")
    
    # Show anomaly details
    print("\nAnomaly Details:")
    anomaly_indices = np.where(anomaly_mask)[0]
    anomaly_types = ["Isolated renal failure (creatinine/urea/K+ up, CRP and WBC normal)",
                     "Occult bleed (hemoglobin and platelets low, inflammation normal)"]
    for i, idx in enumerate(anomaly_indices):
        if i < len(anomaly_types):
            error = reconstruction_error[idx]
            print(f"  {anomaly_types[i]}")
            print(f"    Reconstruction Error: {error:.2f} {'✓ DETECTED' if error > threshold else '✗ MISSED'}")

# Run anomaly detection on all 10 features
plot_anomaly_detection(data, states, anomaly_mask, n_components=2)

Anomaly Detection on 10 Features:
Using 2 principal components to reconstruct 10D data
Threshold: 1.86 (95th percentile of normal data)

Anomaly Detection Performance:
  True Positives  (detected anomalies): 2/2
  False Positives (normal flagged):     7

Anomaly Details:
  Isolated renal failure (creatinine/urea/K+ up, CRP and WBC normal)
    Reconstruction Error: 3.54 ✓ DETECTED
  Occult bleed (hemoglobin and platelets low, inflammation normal)
    Reconstruction Error: 3.93 ✓ DETECTED


---

## Key Takeaways

### How PCA Works (The Algorithm)
1. **Standardize** the data (subtract mean, divide by std dev)
2. **Find PC1**: the direction of maximum variance
3. **Deflate**: remove PC1's contribution, flattening the data
4. **Repeat** for PC2, PC3, etc.

### Why Standardize?
- Variables often have different scales (Platelets in the hundreds vs Potassium around 4)
- Without standardization, larger-scaled variables dominate artificially
- Standardization ensures each test contributes equally to finding patterns

### What PCA Gives You
- **Dimensionality reduction**: 10 lab tests → 2 (or 1) summary scores
- **Cluster visualization**: Patient states become visually separable in 2D
- **Anomaly detection**: Panels that break the usual relationships have high reconstruction error
- **Feature insight**: PC loadings reveal which tests drive each component

### Real-World Applications
- **Summarize a full lab panel** with just a few PCA scores
- **Flag results that don't fit**, before anyone reads the individual numbers
- **Visualize high-dimensional data** in 2D or 3D
- **Reduce noise** by discarding low-variance components

### A Clinical Caveat
A high reconstruction error means "this panel is unusual", not "this patient is sick" - and
certainly not a diagnosis. A rare but entirely benign variant looks just as unusual as a
dangerous one. PCA here is a triage aid for a clinician's attention, nothing more.

### The Core Insight
PCA finds a **new coordinate system** aligned with your data's natural structure. Instead of reading 10 separate numbers, you measure along directions of maximum variation - which often correspond to meaningful patterns like "how acutely unwell is this patient" or "how much has the marrow been suppressed".
